# Fold-Local Joint Reward Targets v1

This notebook regenerates the latest joint reward/checkpoint target recipe as fold-local training targets. It follows the target-scope guard: global `strict_oos` rows are only a source path, and final training rows must be restricted to each train fold with zero eval overlap.

## Experiment Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`
- **Direct comparison baseline**: `joint_reward_target_scope_guard_v1`
- **Role**: `diagnose`
- **Changed layer**: `reward`, `selection`, `sampling`, `model_selection`
- **Target window**: fold-local train windows for `wf_2022_train` and `wf_2023_2024_train`; eval windows are audit-only.
- **Acceptance gate**: target table must be `target_scope=fold_local`, `eval_overlap_rows == 0`, BC label distribution must remain healthy, and any later model must still pass `+30pp / <=15% MaxDD` fixed-split + walk-forward gates.
- **Hypothesis**: the previous joint recipe's useful loss terms can be made trainable by regenerating them only from fold-local train rows, avoiding OOS label leakage.
- **Expected trade-off**: fold-local rows may preserve downtrend and rebound penalties but have fewer recent 2025-2026 rebound clusters than global OOS rows.
- **Failure mode**: if fold-local regeneration removes all rebound targets or degenerates the BC action distribution, do not start PPO.
- **Comparison-table row key**: `fold_local_joint_reward_targets_v1`

## Historical Context

- Closest success: `state_objective_fold_native_teacher_router_v1` produced fold-local targets with clean scope and BC readiness.
- Closest failure: `joint_reward_checkpoint_recipe_v1` built useful joint loss terms but was globally scoped and touched eval dates, so it was blocked by `joint_reward_target_scope_guard_v1`.
- Why this is not a duplicate: this notebook uses the latest `selected_router_no_equity_dd` teacher path and regenerates targets per fold train window, then audits leakage and label distribution before any training handoff.

## External Basis

- Qlib workflow and anti-leakage experiment structure: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL DRL workflow separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfitting caution: https://doi.org/10.21314/JCF.2016.322
- Sparse/hard sample training references for the later BC/PPO branch: Prioritized Experience Replay https://arxiv.org/abs/1511.05952 and Focal Loss https://arxiv.org/abs/1708.02002


In [1]:
from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'src/soxl_quant').exists()), cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.app.state_objective_bc import build_state_objective_bc_dataset_from_frame
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveBCReadinessConfig,
    StateObjectiveTrainingTargetConfig,
    audit_state_objective_target_scope,
    build_fold_local_state_objective_training_targets,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_scope_audit,
    summarize_state_objective_training_targets,
)
from src.soxl_quant.config import load_profile

CARD = ROOT / 'runtime/model_selection/card/fold_local_joint_reward_targets_v1'
SOURCE_ROUTER = ROOT / 'runtime/model_selection/card/state_conditioned_participation_drawdown_objective_v1'
JOINT_RECIPE = ROOT / 'runtime/model_selection/card/joint_reward_checkpoint_recipe_v1'
SCOPE_GUARD = ROOT / 'runtime/model_selection/card/joint_reward_target_scope_guard_v1'
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

def read_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n')

def action_entropy(actions: np.ndarray) -> float:
    if len(actions) == 0:
        return 0.0
    counts = pd.Series(actions).value_counts(normalize=True)
    return float(-(counts * np.log(counts)).sum())

folds = [
    {
        'name': 'wf_2022_train',
        'source_window': 'strict_oos_train_source',
        'train_start': '2020-01-02',
        'train_end': '2021-12-31',
        'eval_start': '2022-01-03',
        'eval_end': '2022-12-30',
    },
    {
        'name': 'wf_2023_2024_train',
        'source_window': 'strict_oos_train_source',
        'train_start': '2020-01-02',
        'train_end': '2022-12-30',
        'eval_start': '2023-01-03',
        'eval_end': '2024-12-31',
    },
]


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Load Latest Teacher Source

The source daily path comes from the selected joint objective teacher, but only train-date rows are kept by the fold-local builder.

In [2]:
joint_selection = read_json(JOINT_RECIPE / 'selection.json')
scope_selection = read_json(SCOPE_GUARD / 'selection.json')
source_selection = read_json(SOURCE_ROUTER / 'selection.json')

teacher_candidate = joint_selection['teacher_candidate']
manifest = pd.read_csv(SOURCE_ROUTER / 'candidate_path_manifest.csv')
source_row = manifest[(manifest['candidate'] == teacher_candidate) & (manifest['window'] == 'strict_oos')].iloc[0]
strict_oos_daily = pd.read_csv(ROOT / source_row['path'])
cluster_metrics = pd.read_csv(SOURCE_ROUTER / 'cluster_metrics.csv')
if 'cluster' not in cluster_metrics.columns and 'cluster_id' in cluster_metrics.columns:
    cluster_metrics = cluster_metrics.rename(columns={'cluster_id': 'cluster'})

print('teacher_candidate:', teacher_candidate)
print('scope_guard_decision:', scope_selection['decision'])
print('strict_oos_rows:', len(strict_oos_daily), strict_oos_daily['datetime'].min(), strict_oos_daily['datetime'].max())
cluster_metrics[cluster_metrics['candidate'] == teacher_candidate].head()


teacher_candidate: selected_router_no_equity_dd
scope_guard_decision: joint_reward_targets_scope_blocked_no_training
strict_oos_rows: 1605 2020-01-02 2026-05-21


,candidate,cluster,rows,start,end,return,buy_and_hold_return,relative_return,max_dd,avg_pos
0,selected_router_no_equity_dd,C002,74,2025-04-04,2025-07-22,42.362882,201.489130,-159.126248,11.084311,0.449880
1,selected_router_no_equity_dd,C003,75,2026-02-04,2026-05-21,31.759225,235.382598,-203.623373,7.259839,0.243563
2,selected_router_no_equity_dd,C001,66,2020-03-18,2020-06-19,60.954643,178.461557,-117.506914,17.169856,0.555591


## Build Fold-Local Joint Targets

This reuses the same joint target terms from the checkpoint recipe: walk-forward relative gap, drawdown gap, rebound participation floor, and downtrend drawdown cap. The output rows are marked `target_scope=fold_local`.

In [3]:
target_config = StateObjectiveTrainingTargetConfig(
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
    cluster_participation_floor=0.35,
    downtrend_position_cap=0.25,
    drawdown_state_threshold=-0.10,
)

fold_targets = build_fold_local_state_objective_training_targets(
    {'strict_oos_train_source': strict_oos_daily},
    folds=folds,
    cluster_metrics=cluster_metrics,
    candidate=teacher_candidate,
    config=target_config,
)

target_summary = summarize_state_objective_training_targets(fold_targets)
fold_summary = (
    fold_targets.groupby('fold', dropna=False)
    .agg(
        rows=('datetime', 'count'),
        unique_dates=('datetime', 'nunique'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        downtrend_rows=('is_downtrend_state', 'sum'),
        max_objective_weight=('objective_weight', 'max'),
        mean_objective_weight=('objective_weight', 'mean'),
        mean_teacher_position=('teacher_position', 'mean'),
        mean_rebound_target_position=('rebound_target_position', 'mean'),
        mean_risk_target_position=('risk_target_position', 'mean'),
    )
    .reset_index()
)

fold_targets.to_csv(CARD / 'fold_local_joint_targets.csv', index=False)
fold_summary.to_csv(CARD / 'fold_target_summary.csv', index=False)
write_json(CARD / 'target_summary.json', target_summary)

print(json.dumps({
    'rows': target_summary['rows'],
    'rebound_rows': target_summary['rebound_rows'],
    'downtrend_rows': target_summary['downtrend_rows'],
    'top_weighted_window': target_summary['top_weighted_window'],
    'required_terms': target_summary['required_terms'],
}, indent=2))
fold_summary


{
  "rows": 1261,
  "rebound_rows": 132,
  "downtrend_rows": 712,
  "top_weighted_window": "wf_2022_train",
  "required_terms": [
    "walkforward_relative_return_gap_penalty",
    "walkforward_drawdown_gap_penalty",
    "high_vol_rebound_participation_floor",
    "downtrend_drawdown_state_cap"
  ]
}


,fold,rows,unique_dates,rebound_rows,downtrend_rows,max_objective_weight,mean_objective_weight,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,wf_2022_train,505,505,66,234,366.115581,228.704969,0.369003,0.374944,0.319272
1,wf_2023_2024_train,756,756,66,478,252.245919,106.255021,0.291014,0.294982,0.257794


## Scope Guard

A trainable target table must have no eval overlap. This is the hard anti-overfit gate introduced after the global joint recipe was blocked.

In [4]:
scope_audit = audit_state_objective_target_scope(fold_targets, folds=folds)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)

scope_audit.to_csv(CARD / 'target_scope_audit.csv', index=False)
write_json(CARD / 'target_scope_summary.json', scope_summary)

print(json.dumps(scope_summary, indent=2))
scope_audit[[
    'window', 'target_scope', 'target_fold', 'fold', 'rows',
    'train_overlap_rows', 'eval_overlap_rows', 'outside_train_eval_rows',
    'leakage_gate', 'scope_issue',
]]


{
  "decision": "state_objective_targets_training_ready",
  "training_allowed": true,
  "leakage_gate": true,
  "audited_groups": 2,
  "blocked_groups": 0,
  "eval_overlap_rows": 0,
  "outside_train_eval_rows": 0,
  "scope_issues": [],
  "top_scope_issue": "",
  "next_mechanism": "run_fold_local_state_objective_training_or_bc"
}


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue
0,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,505,0,0,True,
1,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,


## BC Readiness

Before training, check whether the generated targets produce a non-degenerate action distribution on existing profile action surfaces. This is not model performance; it only decides whether a later weighted BC/PPO run is allowed to start.

In [5]:
feature_frame_path = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
feature_frame = pd.read_pickle(feature_frame_path)
profiles = ['risk_budget_usage_v1', 'balanced', 'balanced_rule_residual_v22']
label_rows = []
for profile_name in profiles:
    profile = load_profile(profile_name)
    dataset = build_state_objective_bc_dataset_from_frame(
        feature_frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    actions = dataset.actions
    if len(actions):
        counts = pd.Series(actions).value_counts()
        dominant_action = int(counts.index[0])
        dominant_share = float(counts.iloc[0] / len(actions))
        unique_actions = int(pd.Series(actions).nunique())
    else:
        dominant_action = -1
        dominant_share = 1.0
        unique_actions = 0
    label_rows.append(
        {
            'profile': profile_name,
            'samples': int(len(actions)),
            'unique_actions': unique_actions,
            'dominant_action': dominant_action,
            'dominant_action_share': dominant_share,
            'action_entropy': action_entropy(actions),
            'obs_dim': int(dataset.obs.shape[1]) if len(dataset.obs.shape) > 1 else 0,
            'rebound_samples': int(dataset.metadata['is_rebound_cluster'].sum()),
            'downtrend_samples': int(dataset.metadata['is_downtrend_state'].sum()),
        }
    )

profile_label_comparison = pd.DataFrame(label_rows)
bc_readiness_scores = score_state_objective_bc_readiness(
    profile_label_comparison,
    target_scope='fold_local',
    config=StateObjectiveBCReadinessConfig(
        max_dominant_action_share=0.90,
        min_unique_actions=3,
        min_samples=100,
    ),
)
bc_summary = summarize_state_objective_bc_readiness(bc_readiness_scores)

profile_label_comparison.to_csv(CARD / 'profile_label_comparison.csv', index=False)
bc_readiness_scores.to_csv(CARD / 'bc_readiness_scores.csv', index=False)
write_json(CARD / 'bc_readiness_summary.json', bc_summary)

print(json.dumps(bc_summary, indent=2))
bc_readiness_scores


{
  "decision": "state_objective_bc_ready_for_training",
  "selected_profile": "risk_budget_usage_v1",
  "selected_target_scope": "fold_local",
  "selected_bc_readiness_gate": true,
  "selected_label_distribution_gate": true,
  "selected_fold_local_scope_gate": true,
  "selected_dominant_action_share": 0.31390728476821195,
  "selected_unique_actions": 5,
  "selected_samples": 755,
  "selected_bc_readiness_loss": 0.0,
  "next_mechanism": "run_fold_local_bc_warm_start_then_walkforward_ppo"
}


,profile,samples,unique_actions,dominant_action,dominant_action_share,action_entropy,obs_dim,rebound_samples,downtrend_samples,sample_gate,label_distribution_gate,fold_local_scope_gate,bc_readiness_gate,bc_readiness_loss,target_scope
0,risk_budget_usage_v1,755,5,2,0.313907,1.482661,315,66,477,True,True,True,True,0.000000,fold_local
1,balanced,755,10,0,0.829139,0.801255,79,66,477,True,True,True,True,0.000000,fold_local
2,balanced_rule_residual_v22,755,5,0,0.923179,0.374393,340,66,477,True,False,True,False,0.231788,fold_local


## Decision

The final decision is only about training handoff readiness. It is still not a runtime promotion and not evidence of meeting the +30pp / <=15% MaxDD performance gate.

In [6]:
training_allowed = bool(scope_summary['training_allowed'] and bc_summary.get('selected_bc_readiness_gate', False))
if training_allowed:
    decision = 'fold_local_joint_reward_targets_ready_for_weighted_bc'
    next_mechanism = 'run_state_objective_weighted_bc_then_walkforward_ppo'
else:
    decision = 'fold_local_joint_reward_targets_not_ready'
    next_mechanism = 'fix_scope_or_action_distribution_before_training'

selection = {
    'experiment': 'fold_local_joint_reward_targets_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'joint_reward_target_scope_guard_v1',
    'role': 'diagnose',
    'changed_layer': ['reward', 'selection', 'sampling', 'model_selection'],
    'source_experiment': 'state_conditioned_participation_drawdown_objective_v1',
    'source_teacher_candidate': teacher_candidate,
    'source_scope_guard_decision': scope_selection['decision'],
    'decision': decision,
    'next_mechanism': next_mechanism,
    'training_allowed': training_allowed,
    'promotion_allowed': False,
    'promotion_blocker': 'Target generation/readiness is not a backtest; promotion still requires fixed-split and walk-forward +30pp / <=15% MaxDD gates.',
    'scope_leakage_gate': bool(scope_summary['leakage_gate']),
    'scope_eval_overlap_rows': int(scope_summary['eval_overlap_rows']),
    'target_rows': int(target_summary['rows']),
    'target_rebound_rows': int(target_summary['rebound_rows']),
    'target_downtrend_rows': int(target_summary['downtrend_rows']),
    'required_terms': [
        'target_position_alignment_loss',
        'walkforward_relative_return_gap_penalty',
        'walkforward_drawdown_gap_penalty',
        'high_vol_rebound_participation_floor',
        'downtrend_drawdown_state_cap',
        'checkpoint_objective_rebound_mae',
        'checkpoint_objective_risk_cap_violation',
    ],
    'selected_profile_for_next_training': bc_summary.get('selected_profile', ''),
    'selected_bc_readiness_gate': bool(bc_summary.get('selected_bc_readiness_gate', False)),
    'selected_dominant_action_share': float(bc_summary.get('selected_dominant_action_share', 1.0)),
    'selected_unique_actions': int(bc_summary.get('selected_unique_actions', 0)),
    'selected_samples': int(bc_summary.get('selected_samples', 0)),
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://arxiv.org/abs/1511.05952',
        'https://arxiv.org/abs/1708.02002',
    ],
    'acceptance_gate_for_next_run': {
        'fold_local_target_scope_required': True,
        'eval_overlap_rows_required': 0,
        'label_distribution_gate_required': True,
        'checkpoint_retention_metric_required': 'checkpoint_objective_loss',
        'rebound_weight_or_focal_loss_allowed': True,
        'relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'max_dd_max_pct': TARGET_MAX_DD,
        'fixed_split_required': True,
        'walkforward_required': True,
        'no_runtime_promotion_from_targets_only': True,
    },
    'sidecars': [
        'fold_local_joint_targets.csv',
        'fold_target_summary.csv',
        'target_summary.json',
        'target_scope_audit.csv',
        'target_scope_summary.json',
        'profile_label_comparison.csv',
        'bc_readiness_scores.csv',
        'bc_readiness_summary.json',
        'summary.json',
    ],
}

summary = {
    'experiment': 'fold_local_joint_reward_targets_v1',
    'decision': decision,
    'training_allowed': training_allowed,
    'selected_profile_for_next_training': selection['selected_profile_for_next_training'],
    'scope_eval_overlap_rows': selection['scope_eval_overlap_rows'],
    'target_rows': selection['target_rows'],
    'target_rebound_rows': selection['target_rebound_rows'],
    'target_downtrend_rows': selection['target_downtrend_rows'],
    'selected_dominant_action_share': selection['selected_dominant_action_share'],
    'selected_unique_actions': selection['selected_unique_actions'],
    'conclusion': [
        f"Fold-local joint targets were generated with {selection['target_rows']} rows and {selection['scope_eval_overlap_rows']} eval-overlap rows.",
        f"BC readiness selected {selection['selected_profile_for_next_training']} with {selection['selected_unique_actions']} actions and dominant share {selection['selected_dominant_action_share']:.3f}.",
        'The next valid step is weighted BC/PPO with checkpoint_objective_loss retention, followed by fixed-split and walk-forward validation; no runtime promotion from target readiness alone.',
    ],
}

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(summary, indent=2))


{
  "experiment": "fold_local_joint_reward_targets_v1",
  "decision": "fold_local_joint_reward_targets_ready_for_weighted_bc",
  "training_allowed": true,
  "selected_profile_for_next_training": "risk_budget_usage_v1",
  "scope_eval_overlap_rows": 0,
  "target_rows": 1261,
  "target_rebound_rows": 132,
  "target_downtrend_rows": 712,
  "selected_dominant_action_share": 0.31390728476821195,
  "selected_unique_actions": 5,
  "conclusion": [
    "Fold-local joint targets were generated with 1261 rows and 0 eval-overlap rows.",
    "BC readiness selected risk_budget_usage_v1 with 5 actions and dominant share 0.314.",
    "The next valid step is weighted BC/PPO with checkpoint_objective_loss retention, followed by fixed-split and walk-forward validation; no runtime promotion from target readiness alone."
  ]
}
